# Vol6 System Executable Pins

This is a **runnable subset** of `Vol6_system_novel_pins.ipynb`.

Policy:
- keep only cells that execute in a clean Python environment
- patch `matplotlib` import (pre-import `matplotlib.artist`)
- wrap heavy demo loops behind `RUN_DEMOS=False`

Kept 5 code cells; quarantined 7.


In [ ]:
RUN_DEMOS = False  # set True to run guarded demo cells


In [ ]:
# SOURCE: MergedNotebook.ipynb#c135
import numpy as np
import math

# -----------------------------
# CORE: nearest-multiple skew + "why" via density around lattice targets
# -----------------------------
def nearest_multiple_sign(phases, H):
    phases = np.asarray(phases, dtype=float)
    phases = phases % 1.0
    m = np.rint(phases / H)                # nearest integer multiple index
    target = (m * H) % 1.0
    above = phases > target
    p = above.mean()
    err = (phases - target + 0.5) % 1.0 - 0.5  # signed error in [-0.5, 0.5)
    return p, int(above.sum()), int((~above).sum()), err

def explain_skew_by_density(phases, H, bins=2000):
    phases = np.asarray(phases, dtype=float) % 1.0
    hist, edges = np.histogram(phases, bins=bins, range=(0.0, 1.0), density=True)
    dx = edges[1] - edges[0]

    # window (bins) sampled on each side of each lattice target
    w = max(2, bins // 1000)  # ~2 bins when bins=2000

    above_mass = 0.0
    below_mass = 0.0

    n_targets = int(np.floor(1.0 / H)) + 2
    for j in range(n_targets):
        t = (j * H) % 1.0
        idx = int(np.clip(t / dx, 0, bins - 1))

        below_slice = hist[max(0, idx - w): idx]
        above_slice = hist[idx: min(bins, idx + w)]

        below_mass += below_slice.sum() * dx
        above_mass += above_slice.sum() * dx

    return above_mass, below_mass, (above_mass - below_mass)

def scan_H(phases, ks=range(6, 31), bins=2000):
    rows = []
    for k in ks:
        H = math.pi / k
        p, a, b, err = nearest_multiple_sign(phases, H)
        above_mass, below_mass, delta = explain_skew_by_density(phases, H, bins=bins)
        rows.append({
            "k": k,
            "H": H,
            "p_above": p,
            "mag": abs(p - 0.5),
            "above": a,
            "below": b,
            "density_delta": delta
        })
    rows.sort(key=lambda r: r["mag"], reverse=True)
    return rows

def pretty(rows, top=10, name=""):
    print(f"\nTOP {top} skew for {name} (nearest-multiple):")
    for r in rows[:top]:
        print(f"pi/{r['k']:2d}  H={r['H']:.6f}  p_above={r['p_above']:.3f}  "
              f"|p-0.5|={r['mag']:.3f}  density_delta={r['density_delta']:+.4f}  "
              f"(N={r['above']+r['below']})")

# -----------------------------
# GENERATORS: ray phases + cube-root phases
# -----------------------------
def gen_ray_vertical_wall_phases(
    L=8.0,
    slope_x=3.0,
    H=math.pi/9,
    N_hits=30000,
    seed=0
):
    """
    Ray in [0,L]^2 with specular reflection.
    Collect y-position (mod 1) when hitting a vertical wall (x=0 or x=L).
    """
    rng = np.random.default_rng(seed)

    # start away from boundaries to avoid trivial hits
    pos = rng.random(2) * (L * 0.999) + (L * 0.0005)

    v = np.array([slope_x, H], dtype=float)
    v = v / np.linalg.norm(v)

    hits = []

    # helper: advance to next wall hit exactly (continuous time)
    while len(hits) < N_hits:
        # times to vertical walls
        if v[0] > 0:
            tx = (L - pos[0]) / v[0]
        else:
            tx = (0 - pos[0]) / v[0]

        # times to horizontal walls
        if v[1] > 0:
            ty = (L - pos[1]) / v[1]
        else:
            ty = (0 - pos[1]) / v[1]

        # choose earliest collision
        t = tx if tx < ty else ty
        nxt = pos + t * v

        # did we hit vertical wall?
        hit_vertical = tx < ty

        if hit_vertical:
            hits.append((nxt[1] % 1.0))
            v[0] *= -1.0
        else:
            v[1] *= -1.0

        # keep position inside box (nudge off boundary)
        pos = np.clip(nxt, 0.0, L)
        eps = 1e-12
        pos = np.where(pos <= 0.0, pos + eps, pos)
        pos = np.where(pos >= L, pos - eps, pos)

    return np.array(hits, dtype=float)

def gen_cube_root_fracs(N=50000, start=2):
    n = np.arange(start, start + N, dtype=float)
    cr = np.cbrt(n)
    frac = cr - np.floor(cr)
    return frac % 1.0

# -----------------------------
# RUN IT
# -----------------------------
H_ref = math.pi / 9
ray_phases  = gen_ray_vertical_wall_phases(L=8.0, slope_x=3.0, H=H_ref, N_hits=30000, seed=1)
cube_phases = gen_cube_root_fracs(N=50000, start=2)

r_rows = scan_H(ray_phases, ks=range(6, 31), bins=2000)
c_rows = scan_H(cube_phases, ks=range(6, 31), bins=2000)

pretty(r_rows, top=10, name="RAY vertical-wall y%1")
pretty(c_rows, top=10, name="CUBE cbrt(n) frac")

# Also show the specific H=pi/9 line for both:
def find_k(rows, k):
    for r in rows:
        if r["k"] == k:
            return r
    return None

k0 = 9
rr = find_k(r_rows, k0)
cc = find_k(c_rows, k0)
print("\nREFERENCE pi/9:")
print(f"RAY : p_above={rr['p_above']:.3f}  |p-0.5|={rr['mag']:.3f}  density_delta={rr['density_delta']:+.4f}")
print(f"CUBE: p_above={cc['p_above']:.3f}  |p-0.5|={cc['mag']:.3f}  density_delta={cc['density_delta']:+.4f}")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c272
# hsf_pid.py
#
# Proportional-Integral-Derivative (PID) controller module for the HSF.
# Implements the Samson v2 feedback control law to minimize phase error.

class SamsonV2_PID:
    """
    A PID controller class that implements the Samson v2 feedback law.
    It calculates a control output to correct for phase error (Δφ).
    """
    def __init__(self, Kp, Ki, Kd, setpoint=0):
        """
        Initializes the PID controller.

        Args:
            Kp (float): Proportional gain.
            Ki (float): Integral gain.
            Kd (float): Derivative gain.
            setpoint (float, optional): The target value for the error. Defaults to 0.
        """
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.setpoint = setpoint
        
        self.integral_term = 0.0
        self.prev_error = 0.0

    def compute(self, phase_error, dt):
        """
        Computes the control output signal based on the phase error.

        Args:
            phase_error (float): The current phase error (Δφ).
            dt (float): The time delta since the last computation.

        Returns:
            float: The control output signal (u_k).
        """
        if dt <= 0:
            return 0.0

        # Proportional term
        p_term = self.Kp * phase_error
        
        # Integral term
        self.integral_term += phase_error * dt
        i_term = self.Ki * self.integral_term
        
        # Derivative term
        derivative = (phase_error - self.prev_error) / dt
        d_term = self.Kd * derivative
        
        # Update previous error for the next iteration
        self.prev_error = phase_error
        
        # The control law output is the sum of the terms
        control_output = p_term + i_term + d_term
        
        return control_output

    def reset(self):
        """Resets the integral and derivative history of the controller."""
        self.integral_term = 0.0
        self.prev_error = 0.0

# --- Unit Tests for hsf_pid.py ---

import unittest

class TestHsfPid(unittest.TestCase):

    def test_pid_components(self):
        # Test with only a P term
        pid = SamsonV2_PID(Kp=10, Ki=0, Kd=0)
        self.assertEqual(pid.compute(phase_error=5, dt=1), 50)

        # Test with only an I term
        pid = SamsonV2_PID(Kp=0, Ki=10, Kd=0)
        pid.compute(phase_error=5, dt=1)
        self.assertEqual(pid.integral_term, 5)
        self.assertEqual(pid.compute(phase_error=2, dt=1), 70) # 10 * (5+2)

        # Test with only a D term
        pid = SamsonV2_PID(Kp=0, Ki=0, Kd=10)
        pid.prev_error = 1 # Set previous error for derivative calculation
        self.assertEqual(pid.compute(phase_error=5, dt=1), 40) # 10 * (5-1)/1

    def test_pid_reset(self):
        pid = SamsonV2_PID(Kp=1, Ki=1, Kd=1)
        pid.compute(phase_error=5, dt=1)
        self.assertNotEqual(pid.integral_term, 0)
        self.assertNotEqual(pid.prev_error, 0)
        
        pid.reset()
        self.assertEqual(pid.integral_term, 0)
        self.assertEqual(pid.prev_error, 0)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

In [ ]:
# SOURCE: MergedNotebook.ipynb#c301
H_RATIO = 0.34906585

# Central attractors - terms that bind the framework
ATTRACTORS = {'harmonic', 'recursive', 'nexus', 'collapse', 'phase', 'state'}

# State transition triad
STATE_TRIAD = {'state', 'phase', 'collapse'}

print(f"H ratio: {H_RATIO}")
print(f"Attractors: {ATTRACTORS}")

In [ ]:
# SOURCE: MergedNotebook.ipynb#c562
# Full code to analyze the harmonic stabilization of the Cosmic Microwave Background (CMB)
# and its relationship to recursive compression and universal harmonics

# Import necessary libraries
import numpy as np

# Constants
planck_constant = 6.62607015e-34  # Joule·seconds
boltzmann_constant = 1.380649e-23  # Joules per Kelvin
speed_of_light = 299792458  # meters per second

# Cosmic Microwave Background (CMB) Peak Frequency Calculation
cmb_temperature = 2.725  # Kelvin
cmb_peak_frequency = (boltzmann_constant * cmb_temperature) / planck_constant

# Harmonic Resonance and Stability Calculation
harmonic_constant = 0.35  # Samson's Law factor

# Calculate the stabilization frequency based on recursive harmonic compression
cmb_harmonic_stabilization = cmb_peak_frequency * harmonic_constant

# Compare with known fundamental quantum resonance states
fine_structure_constant = 1 / 137  # Quantum electromagnetism stability ratio

# Expected fundamental frequency scale by incorporating fine-structure constant
quantum_harmonic_match = cmb_harmonic_stabilization * fine_structure_constant

# Output results
cmb_peak_frequency, cmb_harmonic_stabilization, quantum_harmonic_match


In [ ]:
# SOURCE: MergedNotebook.ipynb#c762
# Re-import necessary libraries
import numpy as np

# Constants
planck_constant = 6.62607015e-34  # Joule·seconds
speed_of_light = 299792458  # meters per second
boltzmann_constant = 1.380649e-23  # Joules per Kelvin

# Cosmic Microwave Background (CMB) Peak Frequency
cmb_temperature = 2.725  # Kelvin
cmb_peak_frequency = (boltzmann_constant * cmb_temperature) / planck_constant

# Harmonic Resonance and Stability Calculation
harmonic_constant = 0.35  # Samson's Law factor

# Calculate the stabilization frequency based on recursive harmonic compression
cmb_harmonic_stabilization = cmb_peak_frequency * harmonic_constant

# Prepare results
cmb_peak_frequency, cmb_harmonic_stabilization


## Quarantined cells (did not run)
| cell | error | note |
|---:|---|---|
| 1 | NameError: name 'List' is not defined | missing dependency / variable not defined in this subset |
| 2 | NameError: name 'List' is not defined | missing dependency / variable not defined in this subset |
| 3 | NameError: name 'Dict' is not defined | missing dependency / variable not defined in this subset |
| 4 | NameError: name 'Tuple' is not defined | missing dependency / variable not defined in this subset |
| 5 | NameError: name 'Dict' is not defined | missing dependency / variable not defined in this subset |
| 8 | ModuleNotFoundError: No module named 'PyInquirer' | optional dependency not installed |
| 11 | NameError: name 'json' is not defined | missing dependency / variable not defined in this subset |
